# 🧹 Notebook 02: Data Cleaning & Engineering
**Goal:** Clean raw data, handle missing values, and prepare a master dataset.

### Steps:
1. Load all raw data
2. Remove NaN rows from air quality
3. Align all datasets to same time period
4. Resample to daily frequency
5. Merge into one master dataframe
6. Save to data/processed/

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8')
print("✅ Libraries ready!")

✅ Libraries ready!


In [2]:
aq_df      = pd.read_csv("data/raw/kathmandu_airquality.csv", parse_dates=["datetime"])
weather_df = pd.read_csv("data/raw/kathmandu_weather.csv", parse_dates=["date"])
trends_df  = pd.read_csv("data/raw/nepal_health_trends.csv", parse_dates=["week"])

print(f"Air Quality : {aq_df.shape}")
print(f"Weather     : {weather_df.shape}")
print(f"Trends      : {trends_df.shape}")

Air Quality : (43848, 8)
Weather     : (1827, 5)
Trends      : (262, 4)


In [3]:
# Check NaN situation before cleaning
print("NaN count per column BEFORE cleaning:")
print(aq_df.isnull().sum())

# Drop rows where pm2_5 is NaN (our most important column)
aq_clean = aq_df.dropna(subset=["pm2_5"])

# Check date range after dropping
print(f"\nDate range after removing NaN:")
print(f"  From : {aq_clean['datetime'].min()}")
print(f"  To   : {aq_clean['datetime'].max()}")
print(f"  Rows : {len(aq_clean)} (removed {len(aq_df) - len(aq_clean)} NaN rows)")

NaN count per column BEFORE cleaning:
datetime                0
pm2_5               22709
pm10                22709
carbon_monoxide     22709
nitrogen_dioxide    22709
ozone               22709
dust                22709
uv_index            22709
dtype: int64

Date range after removing NaN:
  From : 2022-08-04 05:00:00
  To   : 2024-12-31 23:00:00
  Rows : 21139 (removed 22709 NaN rows)


In [4]:
# Set datetime as index for resampling
aq_clean = aq_clean.set_index("datetime")

# Resample hourly → daily averages
aq_daily = aq_clean.resample("D").agg({
    "pm2_5"           : "mean",
    "pm10"            : "mean",
    "carbon_monoxide" : "mean",
    "nitrogen_dioxide": "mean",
    "ozone"           : "mean",
    "dust"            : "mean",
    "uv_index"        : "mean"
}).reset_index()

aq_daily.rename(columns={"datetime": "date"}, inplace=True)

print(f"✅ Resampled to daily: {len(aq_daily)} days")
print(f"Date range: {aq_daily['date'].min()} → {aq_daily['date'].max()}")
print(f"\nFirst few rows:")
aq_daily.head()

✅ Resampled to daily: 881 days
Date range: 2022-08-04 00:00:00 → 2024-12-31 00:00:00

First few rows:


,date,pm2_5,pm10,carbon_monoxide,nitrogen_dioxide,ozone,dust,uv_index
0,2022-08-04,14.884211,21.226316,380.157895,14.226316,82.473684,0.0,1.547368
1,2022-08-05,18.166667,25.933333,387.916667,13.729167,82.666667,0.0,1.904167
2,2022-08-06,20.337500,29.050000,391.166667,13.262500,77.250000,0.0,1.204167
3,2022-08-07,19.170833,27.441667,463.958333,21.091667,64.291667,0.0,2.295833
4,2022-08-08,19.866667,28.416667,524.083333,23.595833,67.708333,0.0,1.710417


In [5]:
# Our air quality starts 2022-08-04 so we align everything to that
start_date = pd.Timestamp("2022-08-04")
end_date   = pd.Timestamp("2024-12-31")

# Trim air quality
aq_daily = aq_daily[
    (aq_daily["date"] >= start_date) & 
    (aq_daily["date"] <= end_date)
].reset_index(drop=True)

# Trim weather
weather_trim = weather_df[
    (weather_df["date"] >= start_date) & 
    (weather_df["date"] <= end_date)
].reset_index(drop=True)

# Trim trends (weekly — keep weeks that fall in range)
trends_trim = trends_df[
    (trends_df["week"] >= start_date) & 
    (trends_df["week"] <= end_date)
].reset_index(drop=True)

print(f"✅ Air Quality : {len(aq_daily)} days")
print(f"✅ Weather     : {len(weather_trim)} days")
print(f"✅ Trends      : {len(trends_trim)} weeks")
print(f"\nAll datasets now cover: {start_date.date()} → {end_date.date()}")

✅ Air Quality : 881 days
✅ Weather     : 881 days
✅ Trends      : 126 weeks

All datasets now cover: 2022-08-04 → 2024-12-31


In [6]:
# Step 1: Merge air quality + weather on date
master_df = pd.merge(aq_daily, weather_trim, on="date", how="left")

print(f"After merging AQ + Weather: {master_df.shape}")

# Step 2: Merge with Google Trends
# Trends is weekly so we forward-fill it to daily
# First create a daily date range and merge trends onto it
trends_daily = trends_trim.rename(columns={"week": "date"})
master_df = pd.merge_asof(
    master_df.sort_values("date"),
    trends_daily.sort_values("date"),
    on="date",
    direction="backward"  # fills each day with the most recent week's value
)

print(f"After merging Trends     : {master_df.shape}")
print(f"\nColumns in master dataset:")
print(list(master_df.columns))
print(f"\nFirst few rows:")
master_df.head()

After merging AQ + Weather: (881, 12)
After merging Trends     : (881, 15)

Columns in master dataset:
['date', 'pm2_5', 'pm10', 'carbon_monoxide', 'nitrogen_dioxide', 'ozone', 'dust', 'uv_index', 'temperature_2m_mean', 'precipitation_sum', 'windspeed_10m_max', 'relative_humidity_2m_mean', 'cough', 'asthma', 'breathing problem']

First few rows:


,date,pm2_5,pm10,carbon_monoxide,nitrogen_dioxide,ozone,dust,uv_index,temperature_2m_mean,precipitation_sum,windspeed_10m_max,relative_humidity_2m_mean,cough,asthma,breathing problem
0,2022-08-04,14.884211,21.226316,380.157895,14.226316,82.473684,0.0,1.547368,22.6,4.5,5.0,89,NaN,NaN,NaN
1,2022-08-05,18.166667,25.933333,387.916667,13.729167,82.666667,0.0,1.904167,22.9,19.3,6.8,87,NaN,NaN,NaN
2,2022-08-06,20.337500,29.050000,391.166667,13.262500,77.250000,0.0,1.204167,23.0,20.3,6.9,87,NaN,NaN,NaN
3,2022-08-07,19.170833,27.441667,463.958333,21.091667,64.291667,0.0,2.295833,22.8,25.9,7.0,88,44.0,11.0,0.0
4,2022-08-08,19.866667,28.416667,524.083333,23.595833,67.708333,0.0,1.710417,22.9,7.5,6.6,88,44.0,11.0,0.0


In [7]:
print("=== MASTER DATASET SUMMARY ===\n")
print(f"Shape       : {master_df.shape}")
print(f"Date range  : {master_df['date'].min().date()} → {master_df['date'].max().date()}")
print(f"\nMissing values per column:")
print(master_df.isnull().sum())
print(f"\nData types:")
print(master_df.dtypes)

=== MASTER DATASET SUMMARY ===

Shape       : (881, 15)
Date range  : 2022-08-04 → 2024-12-31

Missing values per column:
date                         0
pm2_5                        0
pm10                         0
carbon_monoxide              0
nitrogen_dioxide             0
ozone                        0
dust                         0
uv_index                     0
temperature_2m_mean          0
precipitation_sum            0
windspeed_10m_max            0
relative_humidity_2m_mean    0
cough                        3
asthma                       3
breathing problem            3
dtype: int64

Data types:
date                         datetime64[ns]
pm2_5                               float64
pm10                                float64
carbon_monoxide                     float64
nitrogen_dioxide                    float64
ozone                               float64
dust                                float64
uv_index                            float64
temperature_2m_mean                

In [9]:
# Drop breathing problem column (was all zeros — not useful)
master_df = master_df.drop(columns=["breathing problem"], errors="ignore")

# Fix: use .ffill() instead of fillna(method="ffill")
master_df["cough"]  = master_df["cough"].ffill().bfill()
master_df["asthma"] = master_df["asthma"].ffill().bfill()

# Verify no missing values remain
print("Missing values after cleanup:")
print(master_df.isnull().sum())
print(f"\nFinal shape: {master_df.shape}")

Missing values after cleanup:
date                         0
pm2_5                        0
pm10                         0
carbon_monoxide              0
nitrogen_dioxide             0
ozone                        0
dust                         0
uv_index                     0
temperature_2m_mean          0
precipitation_sum            0
windspeed_10m_max            0
relative_humidity_2m_mean    0
cough                        0
asthma                       0
dtype: int64

Final shape: (881, 14)


In [10]:
master_df.to_csv("data/processed/master_dataset.csv", index=False)

print("✅ Master dataset saved to data/processed/master_dataset.csv")
print(f"   Shape     : {master_df.shape}")
print(f"   Date range: {master_df['date'].min().date()} → {master_df['date'].max().date()}")
print(f"   Columns   : {list(master_df.columns)}")

✅ Master dataset saved to data/processed/master_dataset.csv
   Shape     : (881, 14)
   Date range: 2022-08-04 → 2024-12-31
   Columns   : ['date', 'pm2_5', 'pm10', 'carbon_monoxide', 'nitrogen_dioxide', 'ozone', 'dust', 'uv_index', 'temperature_2m_mean', 'precipitation_sum', 'windspeed_10m_max', 'relative_humidity_2m_mean', 'cough', 'asthma']


## ✅ Phase 2 Complete — Master Dataset Ready

| Column | Description |
|--------|-------------|
| date | Daily timestamp |
| pm2_5 | Fine particulate matter (μg/m³) |
| pm10 | Coarse particulate matter (μg/m³) |
| carbon_monoxide | CO levels |
| nitrogen_dioxide | NO2 levels |
| ozone | O3 levels |
| dust | Dust particles |
| uv_index | UV index |
| temperature_2m_mean | Mean daily temperature (°C) |
| precipitation_sum | Daily rainfall (mm) |
| windspeed_10m_max | Max wind speed (km/h) |
| relative_humidity_2m_mean | Mean humidity (%) |
| cough | Google Trends search interest |
| asthma | Google Trends search interest |

**Next: Notebook 03 — Exploratory Data Analysis & Lag Correlation**